# GroupBy Deep Dive — Titanic, FIFA, and IPL (Task 19)

This task notebook is a focused study of one of Pandas' most important operations: **`groupby`**. The idea is simple — split the data by a key, apply some function to each piece, and combine the results — but the variations on that theme (aggregation, transformation, filtering, nested grouping) cover a huge share of real analysis. Three datasets are used as testing grounds, each pushing a different facet of grouping.

The **Titanic** questions start with the basics. Q-1 groups passengers by `Pclass` and reports each class's average age and count of missing ages, using `groupby.groups.keys()` to iterate over groups and `get_group(...)` to inspect each. Q-2 is the subtle one: fill each class's missing `Embarked` values with that class's **mode**, then print value counts. A naive first attempt fails silently — it computes the mode and calls `fillna` without reassigning, so the original frame is never updated. The corrected solutions compare two idioms: a `groupby(...).transform(lambda x: x.fillna(x.mode()[0]))` (vectorised and clean) versus a loop that writes back through `.loc[group_index, 'Embarked']`. Q-3 moves to **nested grouping** — group by `Embarked`, then within each, group by `Pclass` — to produce a dictionary of average fares per class per embarkation port.

The **FIFA World Cup 2022** question, Q-4, introduces a custom function applied per group. You implement Z-normalisation — `(x − μ) / std` — as a `z_normalization(group, cols_to_perform)` function, apply it to four passing/line-break columns via `groupby(...).apply(...)`, and then aggregate each team's minimum, maximum, mean, and standard deviation for a chosen set of columns. It is a practical lesson in **feature engineering inside groups**, where each row is scored relative to its own team's distribution.

Finally the **IPL ball-by-ball** dataset powers four richer questions. Q-5 finds the highest individual score while chasing and the best strike rate (minimum 100 balls faced). Q-6 finds the most successful bowler-batsman pair by dismissals, tie-broken by fewer runs conceded. Q-7 defines a custom batter-pair key (by sorting the two names) to find the highest-scoring batting partnerships, and Q-8 turns that into a full table of pairs with runs, average, and strike rate. Every answer demonstrates that `groupby` combined with `agg`, `transform`, or `apply` — plus a little domain reasoning about cricket — is enough to answer non-trivial questions. The notebook closes with a summary of the reusable patterns.

## 0. Setup

Import `pandas` and `numpy`, then read the Titanic CSV from its published URL.

In [ ]:
import pandas as pd
import numpy as np

url = "https://docs.google.com/spreadsheets/d/e/2PACX-1vQjh5HzZ1N0SU7ME9ZQRzeVTaXaGsV97rU8R7eAcg53k27GTstJp9cRUOfr55go1GRRvTz1NwvyOnuh/pub?gid=1562145139&single=true&output=csv"
titanic_df = pd.read_csv(url)

# 1. Grouping the Titanic Data

`groupby` splits a frame by a key, applies a function to each split, and recombines. The Titanic passenger list — with `Pclass`, `Age`, `Embarked`, and `Fare` — is an ideal playground because its natural groupings (travel class, port of embarkation) map onto real survival and fare differences.

### `Q-1:` Using `groupby` make groups using the `"Pclass"` column and find out the average age and total number of missing values in the `"Age"` column for every group.

## 1.1 Average age and missing counts per class (Q-1)

`titanic_df.groupby('Pclass')` creates a `DataFrameGroupBy` object; the actual groups are reified in `pclass_groups.groups`, a dict mapping each class key to the **index labels** of its rows. Iterating `list(pclass_groups.groups.keys())` therefore visits `1`, `2`, and `3`, and `pclass_groups.get_group(group)` materialises that class's rows as a DataFrame.

Inside the loop, two calculations are printed per class: `['Age'].mean()` (the average age) and `['Age'].isna().sum()` (the count of missing ages). The output is revealing — first class has an average age of 38.2 with 30 missing, second class 29.9 with 11 missing, third class 25.1 with 136 missing. The pattern (wealthier passengers were older; third class had by far the most missing data) is a genuine social insight, and the missing-count column is a data-quality signal: 136 missing ages in third class will heavily affect any age analysis that drops nulls.

A richer alternative is a single aggregated call: `pclass_groups['Age'].agg(['mean', 'count', lambda s: s.isna().sum()])`, which returns the same numbers as a table without a Python loop. The manual iteration is useful for learning what `groupby` holds, but vectorised aggregation is what you would ship.

In [ ]:
# code here
pclass_groups = titanic_df.groupby("Pclass")
for group in list(pclass_groups.groups.keys()):
    print(f"Pclass {group} : Avg Age {pclass_groups.get_group(group)['Age'].mean()} and total missing value {pclass_groups.get_group(group)['Age'].isna().sum()}")

Pclass 1 : Avg Age 38.233440860215055 and total missing value 30
Pclass 2 : Avg Age 29.87763005780347 and total missing value 11
Pclass 3 : Avg Age 25.14061971830986 and total missing value 136


# Task Q-2 — Fill Embarked per Class with the Mode

**Goal:** Group by `Pclass`, fill each group's missing `Embarked` values with **that group's mode**, then print the group value counts in ascending order. The first naive attempt silently fails to update the frame — comparing it with the corrected versions is the lesson.

### `Q-2:` Using `groupby` make groups using the `"Pclass"` column and fill every group's `"Embarked"` column's missing values with the mode value of that group. After that, print every group's `"Embarked"` column's value counts in ascending order.

In [ ]:
# code here - Done in session
for group in list(pclass_groups.groups.keys()):
    mode = pclass_groups.get_group(group)["Embarked"].mode()
    pclass_groups.get_group(group)["Embarked"].fillna(mode)
    print(pclass_groups.get_group(group)["Embarked"].value_counts(ascending = True))

Q      2
C     85
S    127
Name: Embarked, dtype: int64
Q      3
C     17
S    164
Name: Embarked, dtype: int64
C     66
Q     72
S    353
Name: Embarked, dtype: int64


* Nan value is not updated in the dataframe in the above approach.

## 2.1 Why the first attempt failed, and two correct fixes

The original loop did this inside each group:
```python
mode = pclass_groups.get_group(group)['Embarked'].mode()
pclass_groups.get_group(group)['Embarked'].fillna(mode)
```
It computes the mode correctly but calls `fillna` as a *statement* — the returned Series is discarded, and `get_group` returns a fresh object each time, so the fill never reaches `titanic_df`. The value counts still show the `NaN`s untouched. This is a classic Pandas trap: most operations **return a new object**; they do not mutate in place unless you assign the result or pass `inplace=True`.

**Approach 1 — vectorised transform.**
```python
titanic_df1['Embarked'] = titanic_df1.groupby('Pclass')['Embarked'].transform(
    lambda x: x.fillna(x.mode()[0]))
```
`transform` applies the function per group and stitches results back to the original index, so the whole column is filled in one expression. Note `x.mode()[0]` — `mode()` returns a Series (all tied modes), so index the first. The subsequent `grouped.value_counts()` confirms class 1 now has C 85, Q 2, S 129 (the two filled values joining S), class 2 C 17, Q 3, S 164, and class 3 C 66, Q 72, S 353.

**Approach 2 — explicit loop with `.loc`.**
```python
for group in list(pclass_groups.groups.keys()):
    mode_val = pclass_groups.get_group(group)['Embarked'].mode()
    titanic_df2.loc[pclass_groups.get_group(group).index, 'Embarked'] = \
        pclass_groups.get_group(group)['Embarked'].fillna(mode_val[0])
```
Here you compute the filled values and write them back to the correct rows by index using `.loc[...]`. It works, but it is slower and easier to get wrong than `transform`, and it triggers copy-on-write warnings on modern Pandas. Prefer Approach 1.

In [ ]:
# Updated Code : Approach 1
titanic_df1 = titanic_df.copy(True)

# Step 1: Group by 'Pclass' and fill missing values in 'Embarked' with the mode value of that group
titanic_df1['Embarked'] = titanic_df1.groupby('Pclass')['Embarked'].transform(lambda x: x.fillna(x.mode()[0]))

# Step 2: Print the value counts of 'Embarked' for each group in ascending order
grouped = titanic_df1.groupby('Pclass')[['Embarked']]
grouped.value_counts().sort_index(ascending=True)

Pclass  Embarked
1       C            85
        Q             2
        S           129
2       C            17
        Q             3
        S           164
3       C            66
        Q            72
        S           353
dtype: int64

In [ ]:
# Updated Code : Approach 2

titanic_df2 = titanic_df.copy(True)
pclass_groups=titanic_df2.groupby('Pclass')
for group in list(pclass_groups.groups.keys()):
    mode_val=pclass_groups.get_group(group)['Embarked'].mode()
    titanic_df2.loc[pclass_groups.get_group(group).index, 'Embarked'] = pclass_groups.get_group(group)['Embarked'].fillna(mode_val[0])
    print(pclass_groups.get_group(group)['Embarked'].value_counts())

S    129
C     85
Q      2
Name: Embarked, dtype: int64
S    164
C     17
Q      3
Name: Embarked, dtype: int64
S    353
Q     72
C     66
Name: Embarked, dtype: int64


# Task Q-3 — Nested GroupBy (Embarked then Pclass)

**Goal:** Group by `Embarked`; within each port, group by `Pclass`; report the average fare per class per port, rounded to 2 decimals, as a nested dictionary.

### `Q-3:` Make groups based on `"Embarked"` column. And for each of this embarked group, make another group based on `"Pclass"` and find out the average fare (round off up to 2 decimal places) for each "Pclass" for each group of "Embarked".

**Sample Output:**

```bash
{'C': {1: 105, 2: 25, 3: 11},
 'Q': {1: 90, 2: 12, 3: 11},
 'S': {1: 70, 2: 20, 3: 15}}
```

## 3.1 Approach — group of groups

The solution nests two `groupby` operations. The outer `titanic_df.groupby('Embarked')` yields groups `C`, `Q`, and `S`. For each, the code creates a sub-group `embarked_groups.get_group(embarked_group).groupby('Pclass')` whose keys are `1`, `2`, `3`, then fills a two-level dictionary:
```python
my_dict[embarked_group][pclass_group] = round(pclass_groups.get_group(pclass_group)['Fare'].mean(), 2)
```
The result is `{'C': {1: 104.72, 2: 25.36, 3: 11.21}, 'Q': {1: 90.0, 2: 12.35, 3: 11.18}, 'S': {1: 70.36, 2: 20.33, 3: 14.64}}` — note the sample output in the prompt used rounded numbers, while the actual quiet computation is finer.

The pattern 'group by A, then group each A-group by B' is exactly what a **MultiIndex pivot** produces. The idiomatic vectorised equivalent is `titanic_df.groupby(['Embarked', 'Pclass'])['Fare'].mean().round(2)` — a single call returning a two-level index — or `titanic_df.pivot_table(index='Embarked', columns='Pclass', values='Fare', aggfunc='mean')`. The manual nested loop makes the split-apply-combine mechanics explicit, but in production the grouped-and-unnested forms are shorter and faster. Reading the numbers, higher classes paid more from every port, as expected.

In [ ]:
# code here
my_dict = {}

embarked_groups = titanic_df.groupby("Embarked")
for embarked_group in list(embarked_groups.groups.keys()):
    pclass_groups = embarked_groups.get_group(embarked_group).groupby("Pclass")
    my_dict[embarked_group] = {}
    for pclass_group in list(pclass_groups.groups.keys()):
        my_dict[embarked_group][pclass_group] = round(pclass_groups.get_group(pclass_group)["Fare"].mean(), 2)
my_dict

{'C': {1: 104.72, 2: 25.36, 3: 11.21},
 'Q': {1: 90.0, 2: 12.35, 3: 11.18},
 'S': {1: 70.36, 2: 20.33, 3: 14.64}}

# 2. FIFA World Cup — Per-Group Z-Normalisation

This section moves from grouping-and-aggregating to **grouping-and-transforming**: applying a custom function that returns a same-shaped group, adding new normalised columns.

## **Questions Based on Fifa Worldcup - 2022 Dataset:**

You can read the dataset by using the below sample code

```python
import pandas as pd

fifa_df = pd.read_csv("https://docs.google.com/spreadsheets/d/e/2PACX-1vT3D_x_4DS6d51LKJ7ze1sxT5WpV5uiSVOFYHLwBiGru6vFyVv5h5-83AwFjxWYiWfCDjDAaarHAV-k/pub?gid=0&single=true&output=csv")
```

## 2.0 Loading the FIFA data

The FIFA results CSV is read directly from its published Google Sheets URL with `pd.read_csv(...)`. Each row is one team's performance in one match and includes columns such as `Team`, `Passes`, `Passes Completed`, `Attempted Line Breaks`, `Completed Line Breaks`, `Yellow Cards`, and `Possession (%)`. These are cumulative performance statistics, and their raw scale differs wildly between teams (a possession-heavy side attempts far more passes), which is exactly why Q-4 normalises within each team: comparing raw pass counts across teams would mostly reflect opportunity, not efficiency. Keep the row grain in mind — the grouping key `Team` will place several match-rows of the same team together, which is what makes per-team normalisation meaningful.

In [ ]:
import pandas as pd

fifa_df = pd.read_csv("https://docs.google.com/spreadsheets/d/e/2PACX-1vT3D_x_4DS6d51LKJ7ze1sxT5WpV5uiSVOFYHLwBiGru6vFyVv5h5-83AwFjxWYiWfCDjDAaarHAV-k/pub?gid=0&single=true&output=csv")

# Task Q-4 — z_normalization and Group Aggregates

**Goal:** Implement `z_normalization(group, cols_to_perform)`, apply it per team to four passing/line-break columns, then aggregate min/max/mean/std of chosen columns per team.

### `Q-4:` Perform `groupby` based on the `"Team"` column and then perform Z Normalization on top of the below columns of each group:
1. Passes
2. Passes Completed
3. Attempted Line Breaks
4. Completed Line Breaks

You have to make a python function named `z_normalization` which takes two arguments:

1. *group:* Every group that you have created
2. *cols_to_perform:* This parameter takes a list of columns on which you have to perform the Z-Normalization.

For this problem, you have to use th `apply()` method.

$\Large Z \ - \ Normalization = \frac{X_i - \mu}{std} $

After that find out the below values for each group:
- minimum "Passess"
- maximum "Passess"
- minimum "Yellow Cards"
- maximum "Yellow cards"
- average "Yellow Cards"
- maximum "Attempted Line Breaks"
- minimum "Attempted Line Breaks"
- standardard deviation of "Attempted Line Breaks"
- average Possession

## 4.1 Approach — apply a custom function per group

Z-normalisation rescales each value relative to its group's own distribution: `z = (x − μ) / std`. The function loops over the requested columns and, for each, computes the group's `std` and `mean` and writes a new `f"{col}_z_norm"` column back into the group:
```python
def z_normalization(group, cols_to_perform):
    for col in cols_to_perform:
        std = group[col].std()
        mean = group[col].mean()
        group[f'{col}_z_norm'] = (group[col] - mean) / std
    return group
```
Because it **returns a DataFrame of the same shape**, it is a perfect fit for `groupby(...).apply(...)`: `fifa_df.groupby('Team').apply(z_normalization, cols_to_perform=cols_to_perform)` runs it per team and concatenates the results. Calling `.groupby('Team')` again afterwards makes the aggregation clean.

The final `.agg({...})` demonstrates the **dict form** — a mapping of column to a list of functions (`'Passes': ['min','max']`, `'Yellow Cards': ['min','max','mean']`, `'Attempted Line Breaks': ['max','min','std']`, `'Possession (%)': 'mean'`). This produces a two-level column header (column, function) with one row per team, from which you can read each team's passing range and discipline. Two caveats: z-scoring with a group standard deviation fails when a group has zero variance (produces `inf`/`nan`), and `apply` returning the group with new columns is a pattern modern Pandas sometimes warns about in favour of `transform` when only existing columns are rescaled. Still, this is the canonical 'engineer features within groups' recipe.

In [ ]:
# code here
def z_normalization(group, cols_to_perform):
    for col in cols_to_perform:
        std = group[col].std()
        mean = group[col].mean()
        group[f"{col}_z_norm"] = (group[col] - mean)/std
    return group


cols_to_perform = ["Passes", "Passes Completed", "Attempted Line Breaks", "Completed Line Breaks"]
groups = fifa_df.groupby("Team")
groups = groups.apply(z_normalization, cols_to_perform = cols_to_perform).groupby("Team")
groups.agg(
{
    "Passes": ["min", "max"],
    "Yellow Cards": ["min", "max", "mean"],
    "Attempted Line Breaks": ["max", "min", "std"],
    "Possession (%)": "mean"
})

               Passes       Yellow Cards               Attempted Line Breaks  \
                  min   max          min max      mean                   max   
Team                                                                           
Argentina         408   862            0   8  2.285714                   249   
Australia         286   466            0   3  1.750000                   171   
Belgium           512   685            1   3  1.666667                   195   
Brazil            548   695            0   3  1.200000                   193   
Cameroon          295   500            1   5  2.666667                   182   
Canada            448   536            2   4  2.666667                   176   
Costa Rica        231   454            1   3  2.000000                   154   
Croatia           461   724            0   2  1.142857                   259   
Denmark           537   650            1   2  1.666667                   241   
Ecuador           429   484            0

# 3. IPL Ball-by-Ball Analysis

This dataset holds **one row per delivery** — enormous and granular — with columns `ID` (match), `innings`, `batter`, `non-striker`, `bowler`, `batsman_run`, `total_run`, `ballnumber`, `extra_type`, `player_out`, `kind`, and `isWicketDelivery`. The four questions extract batting and bowling insights through grouping and custom keys.

## **Questions on the IPL dataset**

ball by ball dataset - https://drive.google.com/file/d/1-kvv_9KCSAFWcrhS9WgTxSrURkRh6GNt/view?usp=share_link





## 3.0 Loading and understanding the deliveries

`pd.read_csv('ipl_deliveries - ipl_deliveries.csv')` loads every ball of every match. Before analysing, note the semantics: `batsman_run` counts runs credited to the batter (excluding extras like wides), `total_run` counts all runs off the delivery, `ballnumber` identifies the delivery within the innings, and `innings == 2` marks the **chasing** side. `extra_type == 'wides'` marks wide deliveries (no legal ball, so they are conventionally excluded from a batter's balls-faced). These definitions drive every calculation below — and getting them wrong changes the answers, so always confirm what each column actually means.

In [ ]:
balls = pd.read_csv("ipl_deliveries - ipl_deliveries.csv")

# Task Q-5 — Highest Score and Best Strike Rate While Chasing

**Goal:** Find the batter with the highest score while chasing (`innings == 2`), and the best strike rate among chasers who faced at least 100 balls.

### `Q-5:` Find batsman in below category-
* Highest score while chasing
* Best Strike rate while chasing and have faced 100+ balls


> Chasing mean team batting in second inning

## 5.1 Highest chasing score

Filtering to second innings (`df = balls[balls.innings == 2]`) isolates chases. An individual innings score is the sum of a batter's `batsman_run` *within one match*, so the key is the pair `(ID, batter)`: `df.groupby(['ID', 'batter']).sum().sort_values(by='batsman_run', ascending=False)['batsman_run'].head()`. Grouping by both columns collapses each batter's deliveries in each match into a single total, and sorting descending reveals the biggest chases: PC Valthaty 120, V Sehwag 119, SV Samson 119, SR Watson 117, ST Jayasuriya 114.

A subtlety: `.sum()` on the group adds *all* numeric columns, so the selection of `['batsman_run']` afterwards is what makes the output readable. A tidier version would be `df.groupby(['ID','batter'])['batsman_run'].sum().nlargest(5)`. Also note that grouping by `(ID, batter)` is the durable pattern for any 'per-match, per-player' metric — best figures in an innings, most sixes in a match, and so on — and it generalises to `(season, batter)` or `(team, bowler)` by swapping the key.

In [ ]:
# code here
#1
df = balls[balls.innings == 2]
df.groupby(["ID", "batter"]).sum().sort_values(by = "batsman_run", ascending = False)["batsman_run"].head()

ID       batter       
501206   PC Valthaty      120
501243   V Sehwag         119
1254061  SV Samson        119
1136620  SR Watson        117
336018   ST Jayasuriya    114
Name: batsman_run, dtype: int64

## 5.2 Best chasing strike rate (100+ balls)

Strike rate is runs per 100 balls: `batsman_run / balls_faced * 100`. The solution first removes wides (`temp_df = df[~(df.extra_type == 'wides')]`) so that only legal deliveries count against the batter. It then aggregates per batter with the **dict form**:
```python
temp_df = temp_df.groupby('batter').agg({'batsman_run': 'sum', 'ballnumber': 'count'})
temp_df['strike_rate'] = temp_df['batsman_run'] / temp_df['ballnumber'] * 100
temp_df[temp_df['ballnumber'] >= 100].sort_values('strike_rate', ascending=False).reset_index().head()
```
This gives each batter's career chasing runs, balls faced, and derived strike rate, then filters to those with at least 100 balls — a **minimum-sample threshold** that prevents a cameo of 12 runs off 3 balls from topping the chart. The leaders are PJ Cummins (194.7), AD Russell (173.0), LS Livingstone (170.1), SP Narine (168.3), and SO Hetmyer.

Two lessons stand out: computing a *rate* requires both a numerator and a denominator aggregated per group, and any rate leaderboard needs a minimum-denominator filter to be meaningful. The `reset_index()` at the end is cosmetic, turning the grouped index back into a column for a tidy display.

In [ ]:
#2
temp_df = df[~(df.extra_type == "wides")]
temp_df = temp_df.groupby("batter").agg(
{
    "batsman_run": "sum",
    "ballnumber": "count"
})
temp_df["strike_rate"] = temp_df["batsman_run"] / temp_df["ballnumber"] * 100
temp_df[temp_df["ballnumber"] >= 100].sort_values("strike_rate", ascending = False).reset_index().head()


           batter  batsman_run  ballnumber  strike_rate
0      PJ Cummins          222         114   194.736842
1      AD Russell          986         570   172.982456
2  LS Livingstone          182         107   170.093458
3       SP Narine          599         356   168.258427
4      SO Hetmyer          330         200   165.000000

# Task Q-6 — Most Successful Bowler-Batsman Pair

**Goal:** Find the bowler-batsman pair with the most dismissals; break ties by fewer runs conceded by that bowler to that batter. (Run-outs and retirements do not count as bowler dismissals.)

### `Q-6` Most Successful bowler against any batsman. Find that pair of bowler and batsman.
> Most Successful in terms of dissmissal. A bowler who have dissmissed any batsman most no of times. If any two pairs have same no of dissmisal, consider runs conceded by bowler to that batsman. Those who have concede lesser runs is more successful.

## 6.1 Approach — mask the dismissals, group by the pair, sort by two keys

The core trick is constructing a boolean flag for genuine bowler dismissals:
```python
balls['IsBatterOut'] = (balls.batter == balls.player_out) & \
    (~balls.kind.isin(['run out', 'retired hurt', 'retired out']))
```
A batter is truly dismissed by the bowler only when the `batter` equals `player_out` **and** the dismissal `kind` is not a run-out or retirement (which are not credited to the bowler). The `&` with the negated `isin` list encodes that cricket rule precisely.

With the flag in place, grouping by the pair and aggregating does everything needed:
```python
balls.groupby(['bowler','batter']).agg({'IsBatterOut':'sum','batsman_run':'sum'}) \
    .sort_values(by=['IsBatterOut','batsman_run'], ascending=[False, True]).head(10)
```
`IsBatterOut:'sum'` counts how many times the bowler dismissed that batter; `batsman_run:'sum'` totals the runs the batter scored off that bowler. Sorting by `['IsBatterOut', 'batsman_run']` with `ascending=[False, True]` orders by dismissals descending and, among equal dismissal counts, by runs ascending — exactly the tie-break the question specifies. The top rows include Z Khan over MS Dhoni (7 dismissals for 74 runs), Sandeep Sharma over V Kohli (7 for 78), and A Mishra over RG Sharma (7 for 87). The general recipe — engineer a boolean, group by the pair, multi-key sort — transfers to any 'head-to-head record' question.

In [ ]:
# code here

balls["IsBatterOut"] = (balls.batter == balls.player_out)&(~balls.kind.isin(['run out', 'retired hurt', 'retired out']))
balls.groupby(["bowler", "batter"]).agg({
    "IsBatterOut": "sum",
    "batsman_run": "sum"
}).sort_values(by=["IsBatterOut", "batsman_run"], ascending = [False, True]).head(10)

                           IsBatterOut  batsman_run
bowler         batter                              
Z Khan         MS Dhoni              7           74
Sandeep Sharma V Kohli               7           78
A Mishra       RG Sharma             7           87
R Ashwin       RV Uthappa            7          123
SP Narine      RG Sharma             7          137

# Task Q-7 — Most Successful Batting Pair

**Goal:** Find the batting pair that scored the most runs together. The challenge is that a partnership is *unordered* — `(A, B)` and `(B, A)` are the same pair — so a stable key must be built that ignores order.

### `Q-7`: Most successful batting pair in IPL. Batting pair who have scored most runs playing together.


## 7.1 Approach — a canonical pair key

The fix for order-independence is to sort the two names before joining:
```python
def func(x):
    return '-'.join(list(np.sort(x.values)))
balls['batter-pair'] = balls[['batter','non-striker']].apply(func, axis=1)
balls.groupby('batter-pair')['total_run'].sum().sort_values(ascending=False).head()
```
`balls[['batter','non-striker']].apply(func, axis=1)` runs `func` row-wise (`axis=1`), passing each row as a Series; `np.sort` orders the two names alphabetically and `'-'.join` produces a canonical string like `'AB de Villiers-V Kohli'`. Because the key is order-independent, every delivery where either batter faced the other's bowling lands in the same group.

Summing `total_run` per pair gives the highest-scoring partnerships: AB de Villiers–V Kohli (3134), CH Gayle–V Kohli (2802), DA Warner–S Dhawan (2357), G Gambhir–RV Uthappa (1906), KL Rahul–MA Agarwal (1731). The Kohli pairings dominating the top two is a striking result. Key lessons: row-wise `apply(axis=1)` is the right tool for logic that combines two columns, and **sorting to canonicalise an unordered pair** is a reusable trick for any symmetric relationship (marriages, trade partners, handshakes). Note that row-wise apply is slow on millions of rows; a vectorised `np.where(a < b, a + '-' + b, b + '-' + a)` would be faster.

In [ ]:
# code here
def func(x):
    return '-'.join(list(np.sort(x.values)))
balls["batter-pair"] = balls[["batter", "non-striker"]].apply(func, axis = 1)
balls.groupby("batter-pair")["total_run"].sum().sort_values(ascending = False).head()

batter-pair
AB de Villiers-V Kohli    3134
CH Gayle-V Kohli          2802
DA Warner-S Dhawan        2357
G Gambhir-RV Uthappa      1906
KL Rahul-MA Agarwal       1731
Name: total_run, dtype: int64

# Task Q-8 — Full Batting-Pair Table

**Goal:** Produce a DataFrame of all batting pairs with columns `Batsman1`, `Batsman2`, `Runs`, `Avg`, and `StrikeRate`. (Wide balls may be counted for the strike rate, per the note.)

### `Q-8:` Make a dataframe for all batting pairs played together.
```
Batsman1 Batsman2 Runs Avg StrikeRate
```

> Just to ease this question you can count wide-balls for strike rate.

## 8.1 Approach — aggregate, split the key, derive metrics

Building on the `batter-pair` key, aggregate three things per pair: total runs, ball count, and wickets lost:
```python
temp_df = balls.groupby('batter-pair').agg(
    {'total_run':'sum', 'ballnumber':'count', 'isWicketDelivery':'sum'}).reset_index()
```
`reset_index()` moves the pair key back into a regular column so it can be split. `Batsman 1` and `Batsman 2` are then extracted by splitting the canonical string on the hyphen — `temp_df['batter-pair'].apply(lambda x: x.split('-')[0])` and `[1]` — and the run column is renamed for presentation. Finally the derived metrics are computed:
- `StrikeRate = Runs / ballnumber * 100` — runs per 100 balls (the question permits counting wides).
- `Avg = Runs / isWicketDelivery` — the partnership average, runs per wicket lost.

The table is sorted by `Runs` descending, and the named pair columns are selected for the final view. AB de Villiers–V Kohli lead with 3134 runs at a 152.2 strike rate and 44.1 average. Two refinements to keep in mind: a robust average divides by wickets and a partnership with *zero* wickets would give `inf`, so guard with `.replace(0, np.nan)` or add 1; and `isWicketDelivery` genuinely counts deliveries on which a wicket fell, which is not always identical to the number of *partnership-ending* wickets. Still, this question brings together the entire skill set — custom keys, dict aggregation, string splitting, and derived metrics — into one clean table.

In [ ]:
# code here
temp_df = balls.groupby("batter-pair").agg(
{
    "total_run": "sum",
    "ballnumber": "count",
    "isWicketDelivery": "sum"
}).reset_index()
temp_df["Batsman 1"] = temp_df["batter-pair"].apply(lambda x: x.split("-")[0])
temp_df["Batsman 2"] = temp_df["batter-pair"].apply(lambda x: x.split("-")[1])
temp_df.rename(columns = {"total_run": "Runs"}, inplace=True)
temp_df["StrikeRate"] = temp_df["Runs"] / temp_df["ballnumber"] * 100
temp_df["Avg"] = temp_df["Runs"] / temp_df["isWicketDelivery"]
temp_df.sort_values("Runs", ascending = False, inplace=  True)
temp_df[["Batsman 1", "Batsman 2", "Runs", "Avg", "StrikeRate"]]

           Batsman 1        Batsman 2  Runs        Avg  StrikeRate
302   AB de Villiers          V Kohli  3134  44.140845  152.209811
1251        CH Gayle          V Kohli  2802  52.867925  142.017233
1508       DA Warner         S Dhawan  2357  48.102041  136.637681
1954       G Gambhir       RV Uthappa  1906  39.708333  133.754386
2803        KL Rahul       MA Agarwal  1731  52.454545  142.939719
...              ...              ...   ...        ...         ...
350   AD Mascarenhas  Gurkeerat Singh     0   0.000000    0.000000
3552     NLTC Perera         RV Gomez     0   0.000000    0.000000
2270   Iqbal Abdulla         TS Mills     0   0.000000    0.000000
3556      NM Coulter             Nile     0   0.000000    0.000000
2928    Kartik Tyagi   Shashank Singh     0   0.000000    0.000000

[4259 rows x 5 columns]

## Summary

Task 19 was a deep dive into `groupby`. On Titanic you grouped by `Pclass` to get per-class average ages and missing-age counts; learned that `fillna` returns a new object (so the naive loop silently changed nothing) and fixed it with either `transform(lambda x: x.fillna(x.mode()[0]))` or an explicit `.loc` write-back; and performed **nested grouping** (Embarked then Pclass) to build a dictionary of average fares. On FIFA you wrote a `z_normalization(group, cols)` function and applied it per team via `groupby(...).apply(...)`, then aggregated min/max/mean/std with the dict form of `agg`. On IPL ball-by-ball data you found the highest chasing score by grouping on `(ID, batter)`; computed the best chase strike rate with a 100-ball minimum by aggregating runs and balls per batters; identified the most successful bowler-batter pair by engineering an `IsBatterOut` mask for non-run-out dismissals and sorting by `[dismissals, runs conceded]`; and built canonical, order-independent batting-pair keys to rank partnerships and produce a full runs/average/strike-rate table. Recurring motifs: split-apply-combine, custom boolean features, canonical keys for symmetric relationships, and minimum-sample thresholds for any rate comparison.